# FPL exploratory analysis

What the historical data looks like before modelling, and why the planned
decomposition (minutes -> team strength -> attack / defence / saves / bonus /
cards) is shaped the way it is.

Run from the repo root with the venv active. The first run downloads the
archive seasons into `data/fpl/` (git-ignored). Tables only, so it needs
nothing beyond `requirements.txt` + `requirements-fpl.txt`.

In [ ]:
import os, sys
from pathlib import Path
if not Path("fpl").exists() and Path("../fpl").exists():
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
from fpl.ingest import archive, snapshot
from fpl.features import build_feature_table

pd.set_option("display.width", 140, "display.max_columns", 30)
SEASONS = ["2022-23", "2023-24", "2024-25", "2025-26"]
fx = archive.load_seasons(SEASONS)
feats = build_feature_table(fx)
fx.shape, feats.shape

## Coverage
One row per player per fixture; double gameweeks give a player two rows in a GW.

In [ ]:
pgw = feats.groupby("season").agg(player_gws=("element", "size"), players=("element", "nunique"),
                                  gws=("gw", "nunique"),
                                  dgw_rows=("n_fixtures", lambda s: int((s > 1).sum())))
pgw

## Points are mostly zeros
Every listed player counts, including those who never got on. That's why
MAE is low for any model and why ranking metrics (Spearman, top-20) matter
more for team selection.

In [ ]:
pts = feats.groupby("position")["total_points"]
pd.DataFrame({"mean": pts.mean(), "std": pts.std(), "share_zero": pts.apply(lambda s: (s == 0).mean()),
              "p95": pts.quantile(0.95), "max": pts.max()}).round(2)

## Minutes are the first-order driver
P(0 / 1-59 / 60+) per fixture by position, which motivates the multiclass
minutes model. Points conditional on 60+ minutes vary much less than
unconditional points.

In [ ]:
m = fx.assign(bucket=pd.cut(fx["minutes"], [-1, 0, 59, 200], labels=["0", "1-59", "60+"]))
mins = pd.crosstab(m["position"], m["bucket"], normalize="index").round(3)
mins["pts_given_60+"] = m[m["bucket"] == "60+"].groupby("position")["total_points"].mean().round(2)
mins

## How predictive is recent form?
Within-GW Spearman between each simple signal and that GW's points.

In [ ]:
from scipy.stats import spearmanr
def gw_spearman(col):
    vals = [spearmanr(g[col], g["total_points"], nan_policy="omit").statistic
            for _, g in feats.dropna(subset=[col]).groupby(["season", "gw"]) if g[col].nunique() > 1]
    return np.nanmean(vals)
pd.Series({c: gw_spearman(c) for c in ["pts_pf_to_date", "pts_pf_last5", "pts_pf_last3",
                                        "mins_pf_last3", "xgi_pf_last5", "value",
                                        "selected_lag_log", "prev_season_pts_pf"]}).round(3).sort_values()

## xG vs goals
Finishing noise: season-level goals vs xG by position (2022-23 onward).

In [ ]:
fx.groupby(["season", "position"])[["goals_scored", "expected_goals", "assists", "expected_assists"]].sum().round(1)

## Defensive contribution (2025-26 onward)
Share of 60+ minute appearances that reach the threshold (DEF >= 10 CBIT,
MID/FWD >= 12 CBIRT). This is the target of the planned logistic model.

In [ ]:
dc = fx[(fx["season"] >= "2025-26") & (fx["minutes"] >= 60)].copy()
dc["threshold"] = dc["position"].map({"DEF": 10, "MID": 12, "FWD": 12})
dc = dc[dc["threshold"].notna()]
dc.assign(hit=dc["defensive_contribution"] >= dc["threshold"]).groupby("position").agg(
    appearances=("hit", "size"), hit_rate=("hit", "mean"),
    mean_count=("defensive_contribution", "mean")).round(3)

## Home advantage

In [ ]:
fx[fx["minutes"] > 0].groupby(["position", "was_home"])["total_points"].mean().unstack().round(2)

## FPL's own projection (`ep_next`)
Only available for gameweeks snapshotted before the deadline by
`python -m fpl.ingest.snapshot`. The archive's `xP` is written after the
gameweek, so it is dropped on load.

In [ ]:
ep = snapshot.load_ep_next_history()
ep.groupby(["season", "gw"]).size().rename("players") if not ep.empty else "no snapshots yet"

## Baseline backtest results
Written by `python -m fpl.backtest --model baselines`.

In [ ]:
from fpl import config
p = config.BACKTEST_DIR / "metrics_2024-25_2025-26.csv"
pd.read_csv(p).round(3) if p.exists() else f"run the backtest first ({p} missing)"